# Architecting Residual and Depthwise Blocks

Implement three custom PyTorch nn.Module classes from scratch: a ResNet
bottleneck block, a depthwise separable convolution layer, and an
EfficientNetV2 Fused-MBConv layer. No GPU required — all tensors are
small and synthetic.

## Setup

In [ ]:
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)  # for reproducible shape-assertion checks

## Section 1: ResNetBottleneckBlock

Implement a 1x1 squeeze -> 3x3 -> 1x1 expand bottleneck with a residual
skip connection. Include a projection shortcut for when input and
output channel counts differ.

In [ ]:
class ResNetBottleneckBlock(nn.Module):
    def __init__(self, in_channels, mid_channels, out_channels, stride=1):
        super().__init__()

        # TODO: 1x1 convolution that reduces in_channels -> mid_channels
        # (the "squeeze" step), followed by BatchNorm2d(mid_channels)


        # TODO: 3x3 convolution on mid_channels, padding=1, stride=stride,
        # followed by BatchNorm2d(mid_channels)


        # TODO: 1x1 convolution that expands mid_channels -> out_channels
        # (the "expand" step), followed by BatchNorm2d(out_channels)


        self.relu = nn.ReLU()

        # TODO: projection shortcut. If in_channels != out_channels or
        # stride != 1, set self.shortcut to a Sequential of
        # Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)
        # followed by BatchNorm2d(out_channels).
        # Otherwise, set self.shortcut = nn.Identity()


    def forward(self, x):
        # TODO: compute the residual path:
        #   conv1 -> bn1 -> relu -> conv2 -> bn2 -> relu -> conv3 -> bn3
        # then add the shortcut (out = out + self.shortcut(x))
        # and apply a final ReLU before returning
        pass

### Shape-assertion checkpoint — do not modify

In [ ]:
# Same-dimension configuration
block_same = ResNetBottleneckBlock(64, 16, 64)
x1 = torch.randn(2, 64, 32, 32)
out1 = block_same(x1)
assert out1.shape == torch.Size([2, 64, 32, 32]), f"FAIL: got {out1.shape}"
print("PASS: ResNetBottleneckBlock (same dims) output shape:", out1.shape)

# Cross-dimension configuration (requires projection shortcut)
block_diff = ResNetBottleneckBlock(64, 16, 128)
x2 = torch.randn(2, 64, 32, 32)
out2 = block_diff(x2)
assert out2.shape == torch.Size([2, 128, 32, 32]), f"FAIL: got {out2.shape}"
print("PASS: ResNetBottleneckBlock (cross dims, projection) output shape:", out2.shape)

## Section 2: DepthwiseSeparableConv

Implement a depthwise convolution (groups=in_channels) followed by a
pointwise convolution, and report the depthwise stage's parameter count.

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        # TODO: depthwise convolution — nn.Conv2d with kernel_size=3,
        # padding=1, groups=in_channels — followed by BatchNorm2d(in_channels)


        # TODO: pointwise convolution — nn.Conv2d with kernel_size=1,
        # in_channels -> out_channels — followed by BatchNorm2d(out_channels)


        self.relu = nn.ReLU()

    def forward(self, x):
        # TODO: print the depthwise conv's parameter count using
        # sum(p.numel() for p in self.dw_conv.parameters()), then apply
        # dw_conv -> bn1 -> relu -> pw_conv -> bn2 -> relu, and return
        pass

### Parameter comparison (provided)

In [ ]:
in_channels, out_channels = 64, 128
standard_conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
standard_params = sum(p.numel() for p in standard_conv.parameters())
print(f"Standard 3x3 conv ({in_channels}->{out_channels}) params: {standard_params}")

### Shape-assertion checkpoint — do not modify

In [ ]:
dsc = DepthwiseSeparableConv(64, 128)
x3 = torch.randn(2, 64, 32, 32)
out3 = dsc(x3)
assert out3.shape == torch.Size([2, 128, 32, 32]), f"FAIL: got {out3.shape}"
print("PASS: DepthwiseSeparableConv output shape:", out3.shape)

dw_params = sum(p.numel() for p in dsc.dw_conv.parameters())
print(f"Depthwise conv parameter count alone: {dw_params}")
print(f"Reduction vs. standard conv: {round(standard_params/dw_params, 2)}x")

## Section 3: EfficientNetV2MBConv

Implement a Fused-MBConv-style block: a single 3x3 expansion convolution
(fusing what would otherwise be separate depthwise + pointwise expand
steps), SiLU activation, a 1x1 projection convolution, and a residual
connection that only fires when in_channels == out_channels.

In [ ]:
class EfficientNetV2MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, expansion_ratio=4):
        super().__init__()
        expanded = in_channels * expansion_ratio

        # TODO: expansion convolution — nn.Conv2d with kernel_size=3,
        # padding=1, in_channels -> expanded, bias=False — followed by
        # BatchNorm2d(expanded) and nn.SiLU()


        # TODO: projection convolution — nn.Conv2d with kernel_size=1,
        # expanded -> out_channels, bias=False — followed by
        # BatchNorm2d(out_channels)


        # TODO: set self.use_residual to True only when
        # in_channels == out_channels, otherwise False


    def forward(self, x):
        # TODO: apply expand_conv -> bn1 -> act -> project_conv -> bn2,
        # then add the residual (out = out + x) only if self.use_residual
        # is True, and return
        pass

### Shape-assertion checkpoint — do not modify

In [ ]:
# Same-dimension configuration (residual should fire)
mbconv_same = EfficientNetV2MBConv(32, 32, expansion_ratio=4)
x4 = torch.randn(2, 32, 16, 16)
out4 = mbconv_same(x4)
assert out4.shape == torch.Size([2, 32, 16, 16]), f"FAIL: got {out4.shape}"
assert mbconv_same.use_residual == True, "FAIL: residual should fire when in_channels == out_channels"
print("PASS: EfficientNetV2MBConv (same dims, residual fires) output shape:", out4.shape)

# Cross-dimension configuration (residual should NOT fire)
mbconv_diff = EfficientNetV2MBConv(32, 64, expansion_ratio=4)
x5 = torch.randn(2, 32, 16, 16)
out5 = mbconv_diff(x5)
assert out5.shape == torch.Size([2, 64, 16, 16]), f"FAIL: got {out5.shape}"
assert mbconv_diff.use_residual == False, "FAIL: residual should NOT fire when in_channels != out_channels"
print("PASS: EfficientNetV2MBConv (cross dims, no residual) output shape:", out5.shape)

print("\nALL SHAPE-ASSERTION CHECKPOINTS PASSED")